In [1]:
import xarray as xr
import pandas as pd
import numpy as np
import geopandas as gpd
import json
import matplotlib.pyplot as plt
 
from xcube.core.store import new_data_store
from xcube.core.chunk import chunk_dataset
from xcube.core.gridmapping import GridMapping
from xcube.core.geom import mask_dataset_by_geometry
from xcube_resampling.spatial import resample_in_space
from xcube_resampling.gridmapping import GridMapping

In [13]:
INPUT_DIR = "input_irrigation"

In [14]:
irr_store = new_data_store("file", root=INPUT_DIR)

In [4]:
bbox = [-5, 40, 3, 44] # Ebro Basin
# time_range = ("2020-01-01", "2021-12-31")
time_range = ("2020-01-01", "2020-01-31")

In [4]:
store_lccs=new_data_store("s3", root="deep-esdl-public", storage_options=dict(anon=True))

In [9]:
mlds_lc = store_lccs.open_data("LC-1x2025x2025-2.0.0.levels")
lc = mlds_lc.base_dataset

In [11]:
lc = lc.sel(time="2020-01-01")
lc = lc[["crs", "lccs_class"]]
lc

<xarray.Dataset> Size: 8GB
Dimensions:     (lat: 64800, lon: 129600)
Coordinates:
  * lat         (lat) float64 518kB 90.0 90.0 89.99 89.99 ... -89.99 -90.0 -90.0
  * lon         (lon) float64 1MB -180.0 -180.0 -180.0 ... 180.0 180.0 180.0
    time        datetime64[ns] 8B 2020-01-01
Data variables:
    crs         int32 4B ...
    lccs_class  (lat, lon) uint8 8GB dask.array<chunksize=(2025, 2025), meta=np.ndarray>
Attributes: (12/38)
    Conventions:                CF-1.6
    TileSize:                   2025:2025
    cdm_data_type:              grid
    comment:                    
    contact:                    https://www.ecmwf.int/en/about/contact-us/get...
    creation_date:              20181130T095451Z
    ...                         ...
    time_coverage_end:          19921231
    time_coverage_resolution:   P1Y
    time_coverage_start:        19920101
    title:                      Land Cover Map of ESA CCI brokered by CDS
    tracking_id:                61b96fd7-42c3-4374-9de1-0dc3b0bcae2a
    type:                       ESACCI-LC-L4-LCCS-Map-300m-P1Y

In [15]:
%%time
irr_store.write_data(lc, "landcover2020global.zarr")

CPU times: user 18.5 s, sys: 3.36 s, total: 21.9 s
Wall time: 1min 27s


'landcover2020global2.zarr'

In [9]:
irr_store.list_data_ids()

['clms.zarr', 'era5.zarr', 'landcover2020global.zarr']

In [16]:
irr_store.open_data("landcover2020global.zarr")

<xarray.Dataset> Size: 8GB
Dimensions:     (lat: 64800, lon: 129600)
Coordinates:
  * lat         (lat) float64 518kB 90.0 90.0 89.99 89.99 ... -89.99 -90.0 -90.0
  * lon         (lon) float64 1MB -180.0 -180.0 -180.0 ... 180.0 180.0 180.0
    time        datetime64[ns] 8B ...
Data variables:
    crs         int32 4B ...
    lccs_class  (lat, lon) uint8 8GB dask.array<chunksize=(2025, 2025), meta=np.ndarray>
Attributes: (12/38)
    Conventions:                CF-1.6
    TileSize:                   2025:2025
    cdm_data_type:              grid
    comment:                    
    contact:                    https://www.ecmwf.int/en/about/contact-us/get...
    creation_date:              20181130T095451Z
    ...                         ...
    time_coverage_end:          19921231
    time_coverage_resolution:   P1Y
    time_coverage_start:        19920101
    title:                      Land Cover Map of ESA CCI brokered by CDS
    tracking_id:                61b96fd7-42c3-4374-9de1-0dc3b0bcae2a
    type:                       ESACCI-LC-L4-LCCS-Map-300m-P1Y